# Build an image

`ns.Image` builds a local `Dockerfile` on a CPU serverless job with
[kaniko](https://github.com/GoogleContainerTools/kaniko) and pushes it to the
project's `generated-images` Container Registry. Jobs in the same project pull
from that registry without credentials.

There is no setup:

- The first build creates the `generated-images` registry.
- The push uses your own credentials, the ones that create jobs. Each build
  stores a short-lived access token in the `generated-images-push` MysteryBox
  secret and deletes that version when the build ends.

The build context is the Dockerfile alone: `COPY` of local files is not
supported yet, `COPY --from=<image>` works. `RUN` steps share the build
container with the token, so build only Dockerfiles you trust.

In [ ]:
import nebius_serverless as ns

Pass a path (`ns.Image("./Dockerfile")`) or the text (`dockerfile_string=`).
The tag is a hash of the Dockerfile, so a Dockerfile built once is reused.

In [ ]:
image = ns.Image("./Dockerfile")
image

`dry_run=True` shows the reference, whether it would be reused, and the build
job request, without creating anything.

In [ ]:
image.build(dry_run=True)

Build and push, or reuse the image if this tag is already in the registry.
Returns the reference; raises `ns.BuildFailed` if the build job fails.

In [ ]:
ref = await image.build_async()
ref

Run a job on the new image.

In [ ]:
spec = ns.JobSpec(
    ["python", "-c", "import torch; print(torch.__version__)"],
    image=ref,
    gpu="l40s",
    timeout="1h",
)
job = await spec.create_async()
await job.wait_async()
job.display()